# Análise de Dados: Terapia Aquática em Parkinson (PD vs Controlo)

Notebook de apresentação. Descreve os datasets extraídos do pipeline de sensores inerciais,
as fórmulas usadas, e as análises exploratórias feitas até agora comparando o grupo com
Doença de Parkinson (PD) e o grupo de Controlo, com uma avaliação honesta de quais análises
deram um resultado útil e quais não deram.

Estrutura do notebook:
1. Descrição dos datasets e como foram extraídos
2. Fórmulas usadas (segmentação, métricas por ciclo, assimetria, estatística)
3. Definição dos grupos PD e Controlo
4. Análises feitas, da mais simples à mais completa, cada uma com código, resultado e avaliação
5. Síntese final e recomendações

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)


from participants import ANON as _ANON

## 1. Descrição dos datasets e como foram extraídos

### 1.1 Dados brutos
Sensores inerciais AX3 (Axivity), acelerómetro triaxial, 100 Hz, colocados nos pulsos e
tornozelos de cada paciente (`data/sensor_data/{paciente}/{sensor_id}.cwa`). O ficheiro
`data/tempos_protocolo.json` define, para cada paciente/sensor, a janela temporal
(`start_time`/`end_time`) de cada um dos 4 exercícios: `Ex_MS_Bracos` (braços isolados),
`Ex_MI_Pernas` (pernas isoladas), `Ex_Swim_1` e `Ex_Swim_2` (dois episódios de nado, braços
e pernas em simultâneo).

### 1.2 Pipeline de extração (3 granularidades independentes)

| Script | Output | Granularidade | Nº de exemplos |
|---|---|---|---|
| `data_analysis.py` | `Results/{paciente}/{sensor}/{ex}_features.json` | 1 valor por exercício completo (TSFEL sobre o sinal inteiro) | 582 ficheiros, 30 participantes |
| `data_analysis_small_window.py` | `results_small_window/.../..._window_XXXX.json` -> `windows_small.csv` | 1 linha por janela fixa de 250ms | 55.942 linhas |
| `data_analysis_medium.py` | `results_medium_window/.../..._movement_XXXX.json` -> `cycles_medium.csv` | 1 linha por ciclo individual (braçada/pernada) | 5.577 linhas |

### 1.3 Deteção de ciclos (medium window)

Eixo fixo por membro (Y para braços, X para pernas, método Ödek & Özcan, 2023), filtro
passa-baixo Butterworth de 3ª ordem a 2 Hz, e deteção de picos por threshold simples. O
primeiro e o último ciclo de cada exercício são excluídos, por corresponderem à fase de
aceleração/desaceleração e não estarem completos. Ver a secção 2 para as fórmulas exatas.

### 1.4 Datasets agregados usados nas análises deste notebook

| Ficheiro | Conteúdo | Como foi construído |
|---|---|---|
| `cycles_medium.csv` | 1 linha por ciclo, todos os sensores | `analysis_results.py` lê todos os JSONs de `results_medium_window/` |
| `cycles_by_limb.csv` | como acima, mas só sensores corretamente associados a D/E (`build_isolated_summary.py`), com colunas `limb_type` (UL/LL) e `side` (Right/Left) | usa a colocação dos sensores em `data/participants.json` para filtrar só o sensor relevante por membro/exercício |
| `asymmetry_results.csv` | 1 linha por paciente, exercício e membro; índice de assimetria D vs E da força | `analysis_asymmetry.py`, a partir de `summary_small.csv` (agregado das janelas de 250ms) |

### 1.5 Limitações conhecidas destes datasets

- Duração da janela de exercício desigual entre pacientes (algumas 4 a 6 vezes mais longas que a norma para o mesmo exercício nominal).
- SPARC por ciclo individual tem pouca variância (fica sempre perto de -50), o que a torna uma feature fraca a esta granularidade.
- Velocidade é uma estimativa relativa por integração de acelerómetro, não medição direta.
- Só entram em `cycles_by_limb.csv` os sensores com posição (D/E, braço/perna) em `data/participants.json`.

In [ ]:
cyc = pd.read_csv('cycles_medium.csv')
cbl = pd.read_csv('cycles_by_limb.csv')
asym = pd.read_csv('asymmetry_results.csv')

for _df in (cyc, cbl, asym):
    _df['patient'] = _df['patient'].map(_ANON).fillna(_df['patient'])

print('cycles_medium.csv   :', cyc.shape, '|', cyc['patient'].nunique(), 'pacientes')
print('cycles_by_limb.csv  :', cbl.shape, '|', cbl['patient'].nunique(), 'pacientes')
print('asymmetry_results.csv:', asym.shape, '|', asym['patient'].nunique(), 'pacientes')

## 2. Fórmulas usadas

### 2.1 Métricas por ciclo (a partir do segmento de aceleração 3D em unidades g)

$$\text{magnitude}_i = \sqrt{x_i^2+y_i^2+z_i^2}, \qquad \text{dynamic}_i = \text{magnitude}_i - \overline{\text{magnitude}}$$

- **Mechanical Load** (proxy de força): $\sum_i |\text{dynamic}_i|$
- **Peak Acceleration**: $\max_i |\text{dynamic}_i|$
- **RMS Acceleration**: $\sqrt{\tfrac{1}{n}\sum_i \text{dynamic}_i^2}$
- **SPARC** (suavidade, via PSD de Welch): $-\sum \sqrt{(\Delta f)^2 + (\Delta\,\text{amp\_norm})^2}$
- **Velocidade** (estimativa relativa): integração trapezoidal de `dynamic × 9.80665` (g→m/s²), com correção de deriva (detrend linear); `Avg_Velocity = mean(|velocidade|)`

### 2.2 Índice de Assimetria

$$AI(\%) = \frac{X_R - X_L}{X_R + X_L}\times100$$

### 2.3 Variabilidade (Coeficiente de Variação)

$$CV\%(X) = \frac{\sigma(X)}{\mu(X)}\times100$$

### 2.4 Teste de hipótese: t-test de Welch + Cohen's d

$$t = \frac{\bar{X}_{PD}-\bar{X}_{Control}}{\sqrt{s_{PD}^2/n_{PD} + s_{Control}^2/n_{Control}}}, \qquad
d = \frac{\bar{X}_{PD}-\bar{X}_{Control}}{\sqrt{(s_{PD}^2+s_{Control}^2)/2}}$$

$H_0$: não há diferença entre grupos. Interpretação de $d$: ~0.2 pequeno, ~0.5 médio, ~0.8 grande.

In [ ]:
def indice_assimetria(x_right, x_left):
    total = x_right + x_left
    return (x_right - x_left) / total * 100.0 if total != 0 else None

def cv_pct_por_paciente(df, coluna, grupo_map, panel_col='panel'):
    d = df[df['patient'].isin(grupo_map.keys())].copy()
    d['group'] = d['patient'].map(grupo_map)
    agg = d.groupby(['patient','group',panel_col])[coluna].agg(['count','mean','std']).reset_index()
    agg['cv_pct'] = agg['std']/agg['mean']*100
    return agg

def teste_pd_vs_controlo(vals_pd, vals_control):
    pdv, ctv = pd.Series(vals_pd).dropna(), pd.Series(vals_control).dropna()
    if len(pdv) < 2 or len(ctv) < 2:
        return dict(n_pd=len(pdv), n_control=len(ctv), aviso='dados insuficientes')
    t, p = stats.ttest_ind(pdv, ctv, equal_var=False)
    d = (pdv.mean()-ctv.mean())/np.sqrt((pdv.std()**2+ctv.std()**2)/2)
    return dict(n_pd=len(pdv), media_pd=pdv.mean(), n_control=len(ctv), media_control=ctv.mean(), t=t, p=p, cohen_d=d)

## 3. Definição dos grupos PD vs Controlo

Os grupos (PD / Controlo) e as exclusões estão em `data/participants.json` (README, secção 3),
escrito à mão a partir do registo de participantes do estudo; a célula seguinte vai buscá-los a
`participants.py`. Só entra quem tem dados de movimento efetivamente extraídos. A Control 2 está marcada como excluída: os ficheiros `.cwa` dela são bit-a-bit idênticos aos
da Control 1. Incluir as duas seria pseudo-replicação num teste estatístico.

In [ ]:
# Grupos e exclusoes (Control 2: duplicado byte-a-byte do Control 1) em data/participants.json
from participants import PD_ALL, CONTROL_ALL, GROUP_MAP

print(f'PD: n={len(PD_ALL)}   Control: n={len(CONTROL_ALL)}')

cbl = cbl[cbl['patient'].isin(GROUP_MAP.keys())].copy()
cbl['group'] = cbl['patient'].map(GROUP_MAP)
cbl['panel'] = cbl['exercise_label'] + ' - ' + cbl['limb_type'].map({'UL':'Braco','LL':'Perna'})

## 4. Análise 1 (mais simples): médias absolutas por exercício, PD vs Controlo

Pergunta: os grupos diferem na intensidade média (não na variabilidade nem na evolução) de
velocidade, força, SPARC ou duração?

In [ ]:
resultados_medias = []
for panel in cbl['panel'].unique():
    for label, col in [('Velocidade','avg_velocity_ms'),('Forca','mechanical_load'),('SPARC','sparc'),('Duracao','duration_s')]:
        sub = cbl[cbl['panel']==panel].groupby(['patient','group'])[col].mean().reset_index()
        r = teste_pd_vs_controlo(sub[sub['group']=='PD'][col], sub[sub['group']=='Control'][col])
        r.update(panel=panel, metrica=label)
        resultados_medias.append(r)

df_medias = pd.DataFrame(resultados_medias)
df_medias[['panel','metrica','n_pd','media_pd','n_control','media_control','t','p','cohen_d']]

Nenhuma comparação deu p<0.05, e todos os |d| ficam em 0.51 ou menos (21 de 24 abaixo de
0.3). Esta análise não é útil: a intensidade absoluta do movimento não distingue os grupos
nesta amostra, o que nem é surpreendente, já que força e velocidade brutas dependem muito de
fatores não relacionados com a doença, como compleição física, técnica ou esforço percebido.
Não vale a pena aprofundar esta via.

## 5. Análise 2: Variabilidade (CV%) da braçada/pernada, PD vs Controlo

Pedido original do orientador: testar hipótese nula de que a variabilidade da braçada não
difere entre grupos, usando teste-t de Student (aqui usamos Welch, mais robusto a grupos de
tamanho desigual).

In [ ]:
resultados_cv = []
for panel in cbl['panel'].unique():
    for label, col in [('Velocidade','avg_velocity_ms'),('Forca','mechanical_load'),('SPARC','sparc'),('Duracao','duration_s')]:
        agg = cv_pct_por_paciente(cbl[cbl['panel']==panel], col, GROUP_MAP, panel_col='panel')
        r = teste_pd_vs_controlo(agg[agg['group']=='PD']['cv_pct'], agg[agg['group']=='Control']['cv_pct'])
        r.update(panel=panel, metrica=label)
        resultados_cv.append(r)

df_cv = pd.DataFrame(resultados_cv)
df_cv[['panel','metrica','n_pd','media_pd','n_control','media_control','t','p','cohen_d']]

Das 24 comparacoes feitas (6 panels x 4 metricas), **3 atingem p<0.05, todas no mesmo
panel: Swim 1 - Braco**. Forca CV% (d=0.92, p=0.013), SPARC CV% (d=-1.00, p=0.006) e
Duracao CV% (d=1.02, p=0.007) mostram, de forma consistente entre si, maior variabilidade
ciclo-a-ciclo na bracada do grupo PD durante o primeiro episodio de nado -- tres metricas
diferentes, mesmo panel, mesma direcao (mais variabilidade no PD).

Das restantes 21 comparacoes, a mais proxima do limiar e a Duracao CV% em Swim 2 - Braco
(p=0.054, d=0.75), na mesma direcao; as outras 20 (incluindo o mesmo panel testado nas
pernas) nao mostram efeito. Com 24 testes simultaneos, seria de esperar por acaso cerca
de 1 falso positivo a p<0.05 só pelo numero de testes; encontrar 3, no mesmo panel e com a
mesma direcao fisiologica, pesa mais a favor de um sinal real do que de ruido -- mas a
correcao formal para comparacoes multiplas (Bonferroni, alfa=0.05/24=0.002) elimina a
significancia dos tres. Deve ser reportado como um achado a confirmar com um teste dirigido
e amostra maior (testando especificamente a bracada em Swim 1, sem repetir os outros 23
testes), nao como conclusao estabelecida.

Esta via e parcialmente util: nao ha diferenca robusta de variabilidade nos outros 5 panels,
mas o padrao especifico em Swim 1 - Braco e o segundo achado mais interessante desta
analise, depois da assimetria bilateral (Seccao 7).

## 6. Análise 3: Evolução das métricas ao longo do exercício (declive normalizado)

Para cada paciente e exercício, normalizamos a posição do ciclo entre 0 (início) e 1 (fim),
e ajustamos uma regressão linear de cada métrica contra essa posição. O declive indica se a
métrica sobe ou desce ao longo do exercício, o que serve como proxy de fadiga ou de
"sequence effect".

In [ ]:
def slope_normalizado(g, col):
    g = g.sort_values('cycle_number')
    if len(g) < 4: return np.nan
    x = (g['cycle_number']-g['cycle_number'].min())/(g['cycle_number'].max()-g['cycle_number'].min())
    y = g[col].values
    mask = ~np.isnan(y)
    if mask.sum() < 4: return np.nan
    slope, *_ = stats.linregress(x[mask], y[mask])
    return slope

rows = []
for (patient, group, panel), g in cbl.groupby(['patient','group','panel']):
    rows.append({'patient': patient, 'group': group, 'panel': panel, 'n_cycles': len(g),
        'slope_velocity': slope_normalizado(g, 'avg_velocity_ms'),
        'slope_duration': slope_normalizado(g, 'duration_s'),
        'slope_force': slope_normalizado(g, 'mechanical_load'),
        'slope_sparc': slope_normalizado(g, 'sparc')})
slopes = pd.DataFrame(rows)

resultados_slope = []
for panel in slopes['panel'].unique():
    for label, col in [('Velocidade','slope_velocity'),('Forca','slope_force'),('SPARC','slope_sparc'),('Duracao','slope_duration')]:
        sub = slopes[slopes['panel']==panel]
        r = teste_pd_vs_controlo(sub[sub['group']=='PD'][col], sub[sub['group']=='Control'][col])
        r.update(panel=panel, metrica=label)
        resultados_slope.append(r)

df_slope = pd.DataFrame(resultados_slope)
df_slope[['panel','metrica','n_pd','media_pd','n_control','media_control','t','p','cohen_d']]

### Verificação honesta: sinais individuais, não só a média suavizada

A curva de média por *bin* pode parecer mais limpa do que a realidade individual. Antes de
confiar visualmente num padrão, contámos quantos pacientes de cada grupo têm declive
positivo e quantos têm declive negativo:

In [ ]:
for panel in ['Arm exercise - Braco', 'Leg exercise - Perna']:
    print(f'--- {panel} ---')
    for label, col in [('velocidade','slope_velocity'),('forca','slope_force')]:
        sub = slopes[slopes['panel']==panel]
        pdv = sub[sub['group']=='PD'][col]
        ctv = sub[sub['group']=='Control'][col]
        print(f'  {label}: PD {sum(pdv>0)}+/{sum(pdv<0)}-  |  Control {sum(ctv>0)}+/{sum(ctv<0)}-')

Nos Braços, a velocidade está quase dividida a meio dentro do PD (9 positivos, 10
negativos), o que mostra que o gráfico suave exagera visualmente um padrão fraco. A força
tem uma tendência real, mas moderada (PD com 58% negativo, 11 de 19; Controlo com 70% positivo, 7 de 10), não
uma divisão limpa. Nas Pernas não há consistência de sinal em nenhum grupo, o que confirma
que é mesmo ruído, e não um padrão escondido.

Esta análise é parcialmente útil, com uma reserva importante: a força nos Braços tem uma
tendência moderada e clinicamente plausível, compatível com o "sequence effect" da
bradicinesia (declínio de amplitude com repetições sucessivas), mas não é um resultado
limpo nem estatisticamente significativo. Pernas e Swim 2 não mostram nada de aproveitável.
O correto é reportar isto como tendência exploratória, não como conclusão.

## 7. Análise 4: Assimetria bilateral (D vs E), PD vs Controlo

Esta é uma pista clássica na literatura de Parkinson: a doença tende a manifestar-se de
forma assimétrica entre os dois lados do corpo. Testámos o valor absoluto do índice de
assimetria da força (`|asymmetry_index|` em `asymmetry_results.csv`), separando por membro
(braço ou perna) e por contexto (isolado, Swim 1, Swim 2).

In [ ]:
asym = asym[asym['patient'].isin(GROUP_MAP.keys())].copy()
asym['group'] = asym['patient'].map(GROUP_MAP)
asym['abs_ai'] = asym['asymmetry_index'].abs()

label_map = {'Ex_MS_Bracos':'Braco isolado','Ex_MI_Pernas':'Perna isolado'}
def rotulo(row):
    if row['exercise'] in label_map: return label_map[row['exercise']]
    membro = 'Braco' if row['limb_type']=='UL' else 'Perna'
    swim = 'Swim 1' if row['exercise']=='Ex_Swim_1' else 'Swim 2'
    return f'{swim} - {membro}'
asym['panel'] = asym.apply(rotulo, axis=1)

resultados_asym = []
for panel in asym['panel'].unique():
    sub = asym[asym['panel']==panel]
    r = teste_pd_vs_controlo(sub[sub['group']=='PD']['abs_ai'], sub[sub['group']=='Control']['abs_ai'])
    r.update(panel=panel)
    resultados_asym.append(r)

df_asym = pd.DataFrame(resultados_asym)
df_asym[['panel','n_pd','media_pd','n_control','media_control','t','p','cohen_d']]

Na tabela acima, o |AI| é maior no grupo PD do que no Controlo em todos os 6 panels, mas
nenhum panel atinge p<0.05 sozinho. Para além dessa direção, o padrão é menos limpo do que
uma primeira leitura das pernas sugeria:

- não é exclusivo das pernas: os d dos braços ficam dentro do intervalo dos d das pernas;
- o d das pernas não cresce de forma monótona do exercício isolado para Swim 1 e Swim 2;
- o Controlo também sobe nas pernas ao longo dos contextos, por isso parte da aparente
  escalada no PD acompanha uma subida no Controlo.

O maior efeito individual é Swim 2 - Perna, mas como ponto isolado, não como o fim de uma
escalada. Lido de forma simples: um sinal de assimetria pequeno e consistente em direção
(PD > Controlo), sem especificidade anatómica nem escalada monótona, que fica como uma das
pistas exploratórias desta análise (tese, Secção 6.1.4).

Esta análise foi feita só para a assimetria da força (carga mecânica); a assimetria de
velocidade e de SPARC ainda não está calculada em `asymmetry_results.csv`.

## 8. Sintese final: o que e util e o que nao e

| # | Analise | Resultado | Util |
|---|---|---|---|
| 1 | Medias absolutas (intensidade) | Nenhum efeito (p>0.05 e \|d\|≤0.51 em tudo) | Nao |
| 2 | Variabilidade (CV%) da bracada/pernada | 3 de 24 testes com p<0.05 (Swim 1 - Braco: forca, SPARC, duracao, todos d>0.8), mas nao sobrevivem a correcao para comparacoes multiplas | Achado a confirmar |
| 3 | Evolucao ao longo do exercicio (declive) | Tendencia moderada so na forca dos bracos (p=0.067, d=-0.69); pernas e Swim 2 sao ruido | Parcial, com reserva |
| 4 | Assimetria bilateral D vs E | \|AI\| maior no PD nos 6 panels, nenhum com p<0.05; nao e especifico das pernas nem cresce de forma monotona | Pista exploratoria |

### Recomendacao para os proximos passos

1. Aprofundar a assimetria, calculando o \|AI\| para velocidade e SPARC (nao so para forca), e
   ver se a mesma escalada aparece de forma independente nessas metricas.
2. Testar de forma dirigida (sem repetir os outros 23 testes) se a maior variabilidade
   ciclo-a-ciclo da bracada em Swim 1 no grupo PD (Analise 2) se replica numa amostra
   independente -- e o segundo achado mais promissor depois da assimetria.
3. Nao insistir na intensidade absoluta como via principal: foi testada com uma amostra
   razoavel (n=19 PD, n=10 Controlo) e nao mostrou sinal.
4. Ter cuidado com comparacoes multiplas: foram feitas dezenas de testes-t, e qualquer p
   "quase significativo" isolado deve ser tratado como hipotese a confirmar, nao conclusao --
   isto inclui os 3 resultados significativos da Analise 2, que nao sobrevivem a uma correcao
   de Bonferroni.
5. A amostra pequena continua a ser a limitacao central. Qualquer figura ou tabela na tese
   deve reportar o n e ser honesta sobre o poder estatistico limitado.

## 10. Sensibilidade a granularidade da analise (SPARC e carga mecanica)

A secção 2.4 documenta o teste usado em todas as análises acima, sempre sobre features
calculadas **por ciclo** e depois agregadas. O índice de assimetria (Secção 7), porém, não
segue essa regra: é construído a partir de `summary_small.csv`, ou seja, da média de força
mecânica sobre janelas fixas de 250ms cobrindo o exercício inteiro, não de uma média de
cargas por ciclo. Esta diferença de granularidade nunca tinha sido testada diretamente. A
célula seguinte recalcula carga mecânica e SPARC nessa mesma granularidade de 250ms/exercício
inteiro, usando a mesma tabela `PLACEMENT` e os mesmos 6 panels de sempre, e compara com os
resultados por ciclo da Análise 1 (Secção 4).

In [ ]:
from participants import PLACEMENT
from build_isolated_summary import EXERCISE_LIMB_GROUPS

sw = pd.read_csv('summary_small.csv')
sw.columns = [c.lstrip('﻿') for c in sw.columns]
sw['sensor_id'] = sw['sensor'].astype(str).str.split('_').str[0]

_rows = []
for patient, pgroup in sw.groupby('patient'):
    placement = PLACEMENT.get(patient)
    anon = _ANON.get(patient)
    if placement is None or anon is None or anon not in GROUP_MAP:
        continue
    group = GROUP_MAP[anon]
    for exercise, limb_type, label in EXERCISE_LIMB_GROUPS:
        egroup = pgroup[pgroup['exercise'] == exercise]
        if egroup.empty:
            continue
        right_id = placement.get('ULR' if limb_type == 'UL' else 'LLR')
        left_id = placement.get('ULL' if limb_type == 'UL' else 'LLL')
        for side, sensor_id in [('Right', right_id), ('Left', left_id)]:
            if not sensor_id:
                continue
            srows = egroup[egroup['sensor_id'] == sensor_id]
            for _, r in srows.iterrows():
                _rows.append({'patient': anon, 'group': group, 'panel': label,
                               'sparc_mean': r['sparc_mean'], 'mechanical_load_mean': r['mechanical_load_mean']})

sw_long = pd.DataFrame(_rows)
sw_panel = sw_long.groupby(['patient', 'group', 'panel'])[['sparc_mean', 'mechanical_load_mean']].mean().reset_index()

PANEL_DISPLAY = {'Arm': 'Arm exercise - Braco', 'Leg': 'Leg exercise - Perna',
                 'Swim 1 - Leg': 'Swim 1 - Perna', 'Swim 1 - Arm': 'Swim 1 - Braco',
                 'Swim 2 - Leg': 'Swim 2 - Perna', 'Swim 2 - Arm': 'Swim 2 - Braco'}

resultados_granularidade = []
for exercise_label, disp in PANEL_DISPLAY.items():
    sub = sw_panel[sw_panel['panel'] == exercise_label]
    for metrica, col in [('Carga mecanica (janela 250ms)', 'mechanical_load_mean'),
                          ('SPARC (janela 250ms)', 'sparc_mean')]:
        r = teste_pd_vs_controlo(sub[sub['group'] == 'PD'][col], sub[sub['group'] == 'Control'][col])
        r.update(panel=disp, metrica=metrica)
        resultados_granularidade.append(r)

df_granularidade = pd.DataFrame(resultados_granularidade)
df_granularidade[['panel', 'metrica', 'n_pd', 'media_pd', 'n_control', 'media_control', 't', 'p', 'cohen_d']]

## Radar profiles PD vs Control (small, medium, full granularity)

Visual summary of the group comparisons, following the radar of Zia Ur Rehman et al. (2020, *Sensors* 20:6992). Each spoke is one metric in one exercise/limb panel; Control is the reference ring at 0 and PD is plotted around it. Two versions: Cohen's d (same effect size as the thesis tables) and the article's z-score (difference of means / Control SD). Asterisk = p < 0.05 (Welch's t-test).

Script: `radar_values.py`; values saved to `radar_values.csv`. The radar figures themselves are not drawn in this repository.

Extra features beyond the thesis tables: the small window adds energy trend, signal power and dominant frequency (dominant frequency and SPARC are limited by the 250 ms window, 4 Hz resolution); the full window adds TSFEL signal power and fundamental frequency.

**Verdict (useful?):** useful as a reading aid. The radars reproduce the table p-values exactly and make three patterns visible at a glance: the Swim 1 arms variability spike (medium), the global-energy shift across all swim panels (full), and asymmetry above Control in 11 of 12 spokes. New consistent but non-significant pattern: small-window dominant frequency higher in PD in all 6 panels (d 0.53-0.68), not interpretable as cadence.

In [ ]:
import subprocess, sys
import pandas as pd
from IPython.display import display
subprocess.run([sys.executable, 'radar_values.py'], check=True, capture_output=True)
rv = pd.read_csv('radar_values.csv')
display(rv.round(3))